## Clinical Information Extraction with LLMs

This notebook demonstrates a pipeline for extracting structured clinical information
from emergency room (ER) reports using a large language model. It extracts:
- precipitating event (cause of ER visit)
- confidence
- explanation

### Pipeline

1. Clinical text extraction
2. Prompt-based information extraction
3. Structured JSON parsing

In [1]:
import json

from src.data_loader.preprocessing import extract_clinical_text, process_er_folder
from src.information_extraction.extractor import init_extractor, extract_medical_info


In [2]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

init_extractor(MODEL_NAME)


/home/tatiana.bladier/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [3]:
# ── 2. Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a medical information extraction assistant.
Given a medical text, extract structured information and return it as JSON only,
with no preamble or explanation.
"""


In [4]:
# Prompt example

def build_prompt(text: str) -> str:
    return f"""Identify the precipitating event that led to the medical emergency.

Return ONLY JSON:
- event
- confidence
- explanation

Medical text:
{text}
"""

In [5]:
df = process_er_folder("data/raw_data/mtsamples_er/texts", limit=5)
df

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


,event,confidence,explanation,source_file
0,None,unknown,No precipitating event is clearly stated in th...,Abdominal_Pain_-_Consult.txt
1,None,unknown,No precipitating event is clearly stated in th...,Abdominal_Pain_2D_Consult.txt
2,fall off bicycle,high,The patient was admitted to the Emergency Depa...,Abrasions_26_Lacerations_2D_ER_Visit.txt
3,fall off bicycle,high,The patient was admitted to the Emergency Depa...,Abrasions__Lacerations_-_ER_Visit.txt
4,accidental ingestion of medication,high,The patient ingested an unknown amount of Cele...,Accidental_Celesta_Ingestion_-_ER_Visit.txt


In [6]:
df["event"].value_counts()

event
fall off bicycle                      2
accidental ingestion of medication    1
Name: count, dtype: int64

In [7]:
for _, row in df.iterrows():
    print(f"--- {row['source_file']} ---")
    print(json.dumps(row.to_dict(), indent=2))

--- Abdominal_Pain_-_Consult.txt ---
{
  "event": null,
  "confidence": "unknown",
  "explanation": "No precipitating event is clearly stated in the text.",
  "source_file": "Abdominal_Pain_-_Consult.txt"
}
--- Abdominal_Pain_2D_Consult.txt ---
{
  "event": null,
  "confidence": "unknown",
  "explanation": "No precipitating event is clearly stated in the text.",
  "source_file": "Abdominal_Pain_2D_Consult.txt"
}
--- Abrasions_26_Lacerations_2D_ER_Visit.txt ---
{
  "event": "fall off bicycle",
  "confidence": "high",
  "explanation": "The patient was admitted to the Emergency Department after falling off his bicycle, which is the precipitating event leading to the medical emergency.",
  "source_file": "Abrasions_26_Lacerations_2D_ER_Visit.txt"
}
--- Abrasions__Lacerations_-_ER_Visit.txt ---
{
  "event": "fall off bicycle",
  "confidence": "high",
  "explanation": "The patient was admitted to the Emergency Department after falling off his bicycle, which is the precipitating event leading

### Evaluation